# Retrieval-Based Reasoning for FailureSensorIQ

This notebook is written for GitHub upload.

It is intentionally simple and heavily commented.

No Hugging Face login, token, or secret is used.

The notebook does the following:

1. Loads the FailureSensorIQ data.
2. Splits `all.jsonl` into train and test sets.
3. Builds option-level retrieval memory from the train set.
4. Tests TF-IDF, BM25, semantic search, and hybrid search.
5. Computes main test metrics.
6. Computes error analysis.
7. Computes disagreement analysis.
8. Computes robustness metrics on simple and complex perturbed datasets.

In [ ]:
# If you run this notebook on Kaggle, uncomment the next line if packages are missing.
# !pip install -q sentence-transformers openpyxl

In [ ]:
# =========================
# 1. Import packages
# =========================

import os
import json
import math
import time
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity

from sentence_transformers import SentenceTransformer

In [ ]:
# =========================
# 2. Configuration
# =========================

# Random seed for reproducible train/test split.
RANDOM_SEED = 42

# 20% of all.jsonl is used as the test set.
TEST_SIZE = 0.20

# Retrieval score settings.
# For each option, we look for top support examples and top contradiction examples.
TOP_K_SUPPORT = 2
TOP_K_CONTRADICT = 2
CONTRADICT_WEIGHT = 0.75

# Hybrid search setting.
# alpha = 0.1 means:
# 10% TF-IDF score + 90% semantic score.
HYBRID_ALPHA = 0.10

# Semantic embedding model.
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

# Optional sample_50 test set.
# The paper can keep this False if sample_50 is not reported.
RUN_SAMPLE_50 = False

# Output folder.
RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# =========================
# 3. Helper functions for files
# =========================

def find_file(file_name):
    """Find a file in common local and Kaggle folders."""
    search_roots = [
        Path("../data"),
        Path("data"),
        Path("/kaggle/input"),
        Path("/mnt/data"),
        Path("."),
    ]

    for root in search_roots:
        if not root.exists():
            continue
        matches = list(root.rglob(file_name))
        if matches:
            return matches[0]

    raise FileNotFoundError(f"Could not find {file_name}. Put it in data/ or Kaggle input.")


def load_jsonl(path):
    """Load a JSONL file into a pandas DataFrame."""
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return pd.DataFrame(rows)


def get_correct_index(correct_list):
    """Return the index of the True value in the correct list."""
    for i, value in enumerate(correct_list):
        if bool(value):
            return i
    return None


def normalize_dataframe(df):
    """Add common columns needed for evaluation."""
    df = df.copy()

    # Make sure id is integer when possible.
    df["id"] = df["id"].astype(int)

    # Index of correct option.
    df["correct_index"] = df["correct"].apply(get_correct_index)

    # Correct option text.
    df["correct_option"] = df.apply(lambda r: r["options"][r["correct_index"]], axis=1)

    # Correct label, such as A, B, C, D, E or P), Q), etc.
    df["correct_label"] = df.apply(lambda r: r["option_ids"][r["correct_index"]], axis=1)

    # Number of answer options.
    df["n_options"] = df["options"].apply(len)

    # Some perturbed files do not include these metadata columns.
    for col in ["asset_name", "question_type", "relevancy"]:
        if col not in df.columns:
            df[col] = "unknown"
        df[col] = df[col].fillna("unknown")

    return df

In [ ]:
# =========================
# 4. Load datasets
# =========================

main_path = find_file("all.jsonl")
simple_path = find_file("all_10_options_all_perturbed_simple.jsonl")
complex_path = find_file("all_10_options_perturbed_complex.jsonl")

main_df = normalize_dataframe(load_jsonl(main_path))
simple_df = normalize_dataframe(load_jsonl(simple_path))
complex_df = normalize_dataframe(load_jsonl(complex_path))

print("Main data:", main_df.shape)
print("Simple perturbed data:", simple_df.shape)
print("Complex perturbed data:", complex_df.shape)

if RUN_SAMPLE_50:
    sample_path = find_file("sample_50_questions.jsonl")
    sample50_df = normalize_dataframe(load_jsonl(sample_path))
    print("Sample 50 data:", sample50_df.shape)

In [ ]:
# =========================
# 5. Train/test split
# =========================

# We split only all.jsonl.
# The perturbed datasets are filtered using the same test ids.
train_df, test_df = train_test_split(
    main_df,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED,
    shuffle=True,
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

# Keep the test ids for robustness evaluation.
test_ids = set(test_df["id"].tolist())

# Filter perturbed datasets so that they match the original test split.
simple_test_df = simple_df[simple_df["id"].isin(test_ids)].copy().reset_index(drop=True)
complex_test_df = complex_df[complex_df["id"].isin(test_ids)].copy().reset_index(drop=True)

# Add original metadata to perturbed rows where metadata is missing.
metadata_cols = ["asset_name", "question_type", "relevancy"]
id_to_meta = main_df.set_index("id")[metadata_cols].to_dict("index")
for df in [simple_test_df, complex_test_df]:
    for col in metadata_cols:
        df[col] = df.apply(lambda r: id_to_meta.get(r["id"], {}).get(col, r[col]), axis=1)

print("Train size:", len(train_df))
print("Test size:", len(test_df))
print("Simple perturbed test size:", len(simple_test_df))
print("Complex perturbed test size:", len(complex_test_df))

In [ ]:
# =========================
# 6. Build option-level retrieval memory
# =========================

def option_record_text(row, option_text):
    """Create the text used for retrieval."""
    return (
        f"Asset: {row.get('asset_name', 'unknown')}. "
        f"Question type: {row.get('question_type', 'unknown')}. "
        f"Relevancy: {row.get('relevancy', 'unknown')}. "
        f"Question: {row['question']} "
        f"Option: {option_text}"
    )


def build_memory(df):
    """Convert each training option into one retrieval record."""
    records = []

    for _, row in df.iterrows():
        for i, option_text in enumerate(row["options"]):
            records.append({
                "question_id": row["id"],
                "text": option_record_text(row, option_text),
                "option_text": option_text,
                "is_correct": bool(row["correct"][i]),
                "asset_name": row.get("asset_name", "unknown"),
                "question_type": row.get("question_type", "unknown"),
                "relevancy": row.get("relevancy", "unknown"),
            })

    return pd.DataFrame(records)

memory_df = build_memory(train_df)
print("Number of option-level memory records:", len(memory_df))
memory_df.head()

In [ ]:
# =========================
# 7. Simple BM25 implementation
# =========================

class SimpleBM25:
    """A small BM25 implementation to avoid extra dependency."""

    def __init__(self, texts, k1=1.5, b=0.75):
        self.k1 = k1
        self.b = b
        self.docs = [self.tokenize(t) for t in texts]
        self.doc_lengths = np.array([len(d) for d in self.docs])
        self.avgdl = float(np.mean(self.doc_lengths))
        self.N = len(self.docs)

        # Document frequency for each token.
        df = Counter()
        for doc in self.docs:
            for token in set(doc):
                df[token] += 1
        self.df = df

        # Inverse document frequency.
        self.idf = {
            token: math.log(1 + (self.N - freq + 0.5) / (freq + 0.5))
            for token, freq in df.items()
        }

        # Term frequencies for each document.
        self.tfs = [Counter(doc) for doc in self.docs]

    def tokenize(self, text):
        """Lowercase tokenization."""
        return str(text).lower().replace("/", " ").replace("-", " ").split()

    def get_scores(self, query):
        """Return BM25 scores against all memory records."""
        query_tokens = self.tokenize(query)
        scores = np.zeros(self.N, dtype=float)

        for token in query_tokens:
            if token not in self.idf:
                continue
            idf = self.idf[token]
            for i, tf in enumerate(self.tfs):
                freq = tf.get(token, 0)
                if freq == 0:
                    continue
                denom = freq + self.k1 * (1 - self.b + self.b * self.doc_lengths[i] / self.avgdl)
                scores[i] += idf * (freq * (self.k1 + 1)) / denom

        return scores

In [ ]:
# =========================
# 8. Fit retrieval models
# =========================

memory_texts = memory_df["text"].tolist()
memory_is_correct = memory_df["is_correct"].values.astype(bool)

# TF-IDF model.
tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    sublinear_tf=True,
    norm="l2",
    token_pattern=r"(?u)\w+",
)
tfidf_matrix = tfidf_vectorizer.fit_transform(memory_texts)

# BM25 model.
bm25_model = SimpleBM25(memory_texts, k1=1.5, b=0.75)

# Semantic embedding model.
semantic_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
memory_embeddings = semantic_model.encode(
    memory_texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
)

print("Retrieval models are ready.")

In [ ]:
# =========================
# 9. Scoring functions
# =========================

def final_option_score(similarities):
    """Compute support-minus-contradiction retrieval score."""

    support_scores = similarities[memory_is_correct]
    contradict_scores = similarities[~memory_is_correct]

    # Top support similarities.
    if len(support_scores) == 0:
        support = 0.0
    else:
        k = min(TOP_K_SUPPORT, len(support_scores))
        support = float(np.mean(np.sort(support_scores)[-k:]))

    # Top contradiction similarities.
    if len(contradict_scores) == 0:
        contradict = 0.0
    else:
        k = min(TOP_K_CONTRADICT, len(contradict_scores))
        contradict = float(np.mean(np.sort(contradict_scores)[-k:]))

    return support - CONTRADICT_WEIGHT * contradict


def minmax(values):
    """Scale a list of values between 0 and 1."""
    values = np.array(values, dtype=float)
    if values.max() == values.min():
        return np.zeros_like(values)
    return (values - values.min()) / (values.max() - values.min())


def score_options_for_row(row):
    """Return option scores for all retrieval methods for one question."""

    tfidf_scores = []
    bm25_scores = []
    semantic_scores = []

    # Score each option independently.
    for option_text in row["options"]:
        query_text = option_record_text(row, option_text)

        # TF-IDF similarities.
        q_tfidf = tfidf_vectorizer.transform([query_text])
        tfidf_sim = cosine_similarity(q_tfidf, tfidf_matrix).ravel()
        tfidf_scores.append(final_option_score(tfidf_sim))

        # BM25 similarities.
        bm25_sim = bm25_model.get_scores(query_text)
        bm25_scores.append(final_option_score(bm25_sim))

        # Semantic similarities.
        q_emb = semantic_model.encode([query_text], normalize_embeddings=True, show_progress_bar=False)[0]
        sem_sim = memory_embeddings @ q_emb
        semantic_scores.append(final_option_score(sem_sim))

    # Hybrid score.
    # We combine normalized TF-IDF and semantic option scores.
    tfidf_norm = minmax(tfidf_scores)
    semantic_norm = minmax(semantic_scores)
    hybrid_scores = HYBRID_ALPHA * tfidf_norm + (1 - HYBRID_ALPHA) * semantic_norm

    return {
        "TF-IDF": np.array(tfidf_scores),
        "BM25": np.array(bm25_scores),
        "Semantic search": np.array(semantic_scores),
        "Hybrid search": np.array(hybrid_scores),
    }


def predict_row(row):
    """Predict one row with all retrieval methods."""
    score_dict = score_options_for_row(row)
    results = {}

    for method_name, scores in score_dict.items():
        pred_index = int(np.argmax(scores))
        results[method_name] = {
            "pred_index": pred_index,
            "pred_label": row["option_ids"][pred_index],
            "pred_option": row["options"][pred_index],
            "score": float(scores[pred_index]),
        }

    return results

In [ ]:
# =========================
# 10. Evaluate one dataset
# =========================

def evaluate_dataset(df, dataset_name):
    """Evaluate all retrieval methods on a dataframe."""
    rows = []
    start_time = time.time()

    for _, row in df.iterrows():
        predictions = predict_row(row)

        for method_name, pred in predictions.items():
            is_correct = pred["pred_index"] == row["correct_index"]
            rows.append({
                "dataset": dataset_name,
                "method": method_name,
                "id": row["id"],
                "asset_name": row.get("asset_name", "unknown"),
                "question_type": row.get("question_type", "unknown"),
                "relevancy": row.get("relevancy", "unknown"),
                "n_options": row["n_options"],
                "correct_index": row["correct_index"],
                "correct_label": row["correct_label"],
                "pred_index": pred["pred_index"],
                "pred_label": pred["pred_label"],
                "correct": is_correct,
            })

    elapsed = time.time() - start_time
    result_df = pd.DataFrame(rows)

    # Add average time per question per method.
    n_questions = len(df)
    n_methods = result_df["method"].nunique()
    time_per_question = elapsed / max(n_questions * n_methods, 1)
    result_df["time_per_question_sec"] = time_per_question

    return result_df

main_results = evaluate_dataset(test_df, "Original_Test")
simple_results = evaluate_dataset(simple_test_df, "SimplePert")
complex_results = evaluate_dataset(complex_test_df, "ComplexPert")

print("Finished evaluation.")
main_results.head()

In [ ]:
# =========================
# 11. Main metrics table
# =========================

def summary_table(result_df):
    """Create accuracy table."""
    summary = []

    for method, group in result_df.groupby("method"):
        n = len(group)
        correct = int(group["correct"].sum())
        wrong = n - correct
        acc = 100 * correct / n
        t = float(group["time_per_question_sec"].iloc[0])

        summary.append({
            "Method": method,
            "N": n,
            "Accuracy (%)": round(acc, 2),
            "Correct": correct,
            "Wrong": wrong,
            "Invalid rate (%)": 0.0,
            "Inference time / question (s)": round(t, 4),
        })

    return pd.DataFrame(summary).sort_values("Accuracy (%)", ascending=False)

main_summary = summary_table(main_results)
main_summary

In [ ]:
# =========================
# 12. Error analysis
# =========================

# Accuracy by question type.
error_by_question_type = (
    main_results
    .groupby(["method", "question_type"])["correct"]
    .mean()
    .mul(100)
    .round(2)
    .reset_index()
)

# Accuracy by number of answer options.
error_by_options = (
    main_results
    .groupby(["method", "n_options"])["correct"]
    .mean()
    .mul(100)
    .round(2)
    .reset_index()
)

# Accuracy by asset type.
error_by_asset = (
    main_results
    .groupby(["asset_name", "method"])["correct"]
    .mean()
    .mul(100)
    .round(2)
    .reset_index()
)

print("Error by question type")
display(error_by_question_type)

print("Error by number of options")
display(error_by_options)

print("Error by asset")
display(error_by_asset)

In [ ]:
# =========================
# 13. Disagreement analysis
# =========================

def disagreement(result_df, method_a, method_b):
    """Compare two methods on the same questions."""
    a = result_df[result_df["method"] == method_a][["id", "correct"]].rename(columns={"correct": "a_correct"})
    b = result_df[result_df["method"] == method_b][["id", "correct"]].rename(columns={"correct": "b_correct"})
    merged = a.merge(b, on="id")

    both_correct = int(((merged.a_correct == True) & (merged.b_correct == True)).sum())
    both_wrong = int(((merged.a_correct == False) & (merged.b_correct == False)).sum())
    a_correct_b_wrong = int(((merged.a_correct == True) & (merged.b_correct == False)).sum())
    a_wrong_b_correct = int(((merged.a_correct == False) & (merged.b_correct == True)).sum())

    return {
        "Comparison": f"{method_a} vs {method_b}",
        "Both correct": both_correct,
        "Both wrong": both_wrong,
        "First correct, second wrong": a_correct_b_wrong,
        "First wrong, second correct": a_wrong_b_correct,
    }

comparisons = [
    ("TF-IDF", "Semantic search"),
    ("BM25", "Semantic search"),
    ("Hybrid search", "Semantic search"),
]

disagreement_table = pd.DataFrame([disagreement(main_results, a, b) for a, b in comparisons])
disagreement_table

In [ ]:
# =========================
# 14. Robustness table
# =========================

def accuracy_by_method(result_df, name):
    """Return accuracy by method as a dictionary."""
    return (
        result_df
        .groupby("method")["correct"]
        .mean()
        .mul(100)
        .round(2)
        .rename(name)
    )

orig_acc = accuracy_by_method(main_results, "Acc@Original (%)")
simple_acc = accuracy_by_method(simple_results, "Acc@SimplePert (%)")
complex_acc = accuracy_by_method(complex_results, "Acc@ComplexPert (%)")

# Consistency means the same id is correct on both original and complex perturbed version.
consistency_rows = []
for method in main_results["method"].unique():
    orig = main_results[main_results["method"] == method][["id", "correct"]].rename(columns={"correct": "orig_correct"})
    comp = complex_results[complex_results["method"] == method][["id", "correct"]].rename(columns={"correct": "complex_correct"})
    merged = orig.merge(comp, on="id")
    consist = 100 * ((merged["orig_correct"] == True) & (merged["complex_correct"] == True)).mean()
    consistency_rows.append({"method": method, "Acc@Consist-Complex (%)": round(consist, 2)})

consistency = pd.DataFrame(consistency_rows).set_index("method")

robustness_table = pd.concat([orig_acc, simple_acc, complex_acc, consistency], axis=1).reset_index()
robustness_table = robustness_table.rename(columns={"method": "Method"})
robustness_table

In [ ]:
# =========================
# 15. Save all results
# =========================

main_summary.to_csv(RESULTS_DIR / "retrieval_main_summary.csv", index=False)
main_results.to_csv(RESULTS_DIR / "retrieval_main_predictions.csv", index=False)
robustness_table.to_csv(RESULTS_DIR / "retrieval_robustness_summary.csv", index=False)
disagreement_table.to_csv(RESULTS_DIR / "retrieval_disagreement.csv", index=False)

with pd.ExcelWriter(RESULTS_DIR / "retrieval_error_analysis.xlsx") as writer:
    main_summary.to_excel(writer, sheet_name="main_summary", index=False)
    error_by_question_type.to_excel(writer, sheet_name="by_question_type", index=False)
    error_by_options.to_excel(writer, sheet_name="by_num_options", index=False)
    error_by_asset.to_excel(writer, sheet_name="by_asset", index=False)
    disagreement_table.to_excel(writer, sheet_name="disagreement", index=False)
    robustness_table.to_excel(writer, sheet_name="robustness", index=False)

print("Saved result files to", RESULTS_DIR.resolve())